[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/colab_homography_propagation.ipynb)

# Homography propagation — acceptance + `max_gap` calibration (Phase 3.5a)

Calibrates and validates the propagation stage on the bake-off clip, against the
Stage-1 checkpoint (`keypoints.parquet` + `trajectories_px.parquet`). CPU-only.

**Method (held-out, ground-truthed):** drop every 5th landmark anchor, propagate
from the rest, and score the propagated homography at each held-out anchor
against its own detected landmarks (pitch-unit reprojection error). Sweep
`max_gap` to find the largest window whose held-out median error stays ≤ 0.05.

**Acceptance gate:**
1. A `max_gap` exists with held-out **median error ≤ 0.05**.
2. **Coverage** at that `max_gap` is materially above the 16% anchor baseline.

The chosen `max_gap` becomes the `propagate_homographies` / `build_homographies`
default (currently 25). Note: the coverage printed here is from the held-out run
(every 5th anchor removed), so it slightly *under*-reports the true coverage of a
full `analyze_video` run — it is directional for picking the knee.

In [ ]:
# Install (CPU runtime is fine — no GPU needed).
!rm -rf /content/soccer-vision
!git clone -q https://github.com/PatrickJReed/soccer-vision.git /content/soccer-vision
!pip install -q "/content/soccer-vision/packages/soccer-vision[roboflow]"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS, build_frame_homographies
from soccer_vision.pitch.propagation import propagate_homographies

OUT = Path("/content/out")
if not (OUT / "keypoints.parquet").exists():
    OUT = Path("/content/drive/MyDrive/soccer-vision/out")
CLIP = "/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4"

kp = pd.read_parquet(OUT / "keypoints.parquet")
traj = pd.read_parquet(OUT / "trajectories_px.parquet")
anchors = build_frame_homographies(kp, conf_threshold=0.5)
total = int(traj["frame"].max()) + 1
print(f"checkpoint: {OUT}")
print(f"anchors: {len(anchors)} ({len(anchors) / total:.1%} of {total} frames)")

In [ ]:
import cv2, numpy as np, pandas as pd, itertools
from pathlib import Path
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS, build_frame_homographies
from soccer_vision.pitch.propagation import compute_interframe_homographies, propagate_homographies

OUT = Path("/content/out")
if not (OUT / "keypoints.parquet").exists():
    OUT = Path("/content/drive/MyDrive/soccer-vision/out")
CLIP = "/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4"
kp = pd.read_parquet(OUT / "keypoints.parquet")
traj = pd.read_parquet(OUT / "trajectories_px.parquet")
anchors = build_frame_homographies(kp, conf_threshold=0.5)
total = int(traj["frame"].max()) + 1

keys = sorted(anchors)
held = keys[::5]
train = {f: H for f, H in anchors.items() if f not in held}

MAX = 60
pairs: set[int] = set()
for a, b in itertools.pairwise(sorted(train)):
    if 1 <= b - a - 1 <= MAX:
        pairs.update(range(a, b))

cap = cv2.VideoCapture(CLIP)
pos = 0
def read_frame(idx):
    global pos
    if idx < pos:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); pos = idx
    while pos < idx:
        if not cap.grab(): return None
        pos += 1
    ok, fr = cap.read(); pos += 1
    return fr if ok else None

interframe = compute_interframe_homographies(read_frame, pairs, traj, downscale=0.5)  # ONE pass
cap.release()

def reproj_error(H, frame):
    kb = kp[(kp.frame == frame) & (kp.conf >= 0.5) & (kp.kp_idx < len(PITCH_LANDMARKS))]
    if len(kb) < 4: return None
    pts = np.column_stack([kb.x_px, kb.y_px, np.ones(len(kb))])
    m = (H @ pts.T).T; m /= m[:, 2:3]
    return float(np.linalg.norm(m[:, :2] - PITCH_LANDMARKS[kb.kp_idx.to_numpy()], axis=1).mean())

print(f"{'max_gap':>8} {'coverage':>9} {'held median err':>16} {'n':>5}")
for mg in [10, 20, 30, 45, 60]:
    out = propagate_homographies(train, interframe, max_gap=mg)
    errs = sorted(e for e in (reproj_error(out[f].H, f) for f in held if f in out) if e is not None)
    med = errs[len(errs) // 2] if errs else float("nan")
    print(f"{mg:8d} {len(out) / total:9.1%} {med:16.3f} {len(errs):5d}")


**Video read:** the video is read **once** (downscaled ORB, sequential grab/retrieve), producing a precomputed inter-frame map. The `max_gap` sweep then runs as pure matrix composition (seconds), so the whole sweep completes in a few minutes instead of hours.

## Interpreting the sweep

- Pick the **largest `max_gap`** whose held-out median error is **≤ 0.05** — that
  is the calibrated default (set it on `propagate_homographies` /
  `build_homographies`; currently 25).
- The coverage column at that `max_gap` is the lift over the ~16% anchor
  baseline (and is a slight under-estimate — see the header note). Confirm it
  clears the bar by re-running `analyze_video` on the clip with the chosen
  `max_gap` and reading its `PipelineResult.homography_coverage` /
  `propagated_coverage`.
- If error climbs past 0.05 even at small `max_gap`, propagation is fragile on
  this clip → revisit (denser anchors via the pitch-detector fine-tune, 3.5b).
- If `n` (held-out frames scored) is small, the every-5th hold-out left few
  isolated anchors — the estimate is noisier but still directional.